# Setup

In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [2]:
len(documents)

72

# Generating ground truth

In [3]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

# Q1. Generating questions

In [4]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

token = os.getenv("GEMINI_API_KEY")
endpoint = "https://generativelanguage.googleapis.com/v1beta/openai/"
model='gemini-2.5-flash'

openai_client = OpenAI(
    base_url=endpoint,
    api_key=token,
)

In [5]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [6]:
import json
from evaluation_utils import llm_structured

In [34]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["filename"]
        })

    return results, usage

In [37]:
ground_truth = []
usages = []

for i in range(3):
    print(f"processing document \'{documents[i]['filename']}\' input tokens")

    results, usage = generate_ground_truth(documents[i])

    ground_truth.extend(results)
    usages.append(usage)

processing document '01-agentic-rag/lessons/01-intro.md' input tokens
processing document '01-agentic-rag/lessons/02-environment.md' input tokens
processing document '01-agentic-rag/lessons/03-rag.md' input tokens


In [46]:
ground_truth[0]

{'question': "What's the main system we're going to put together in this module?",
 'document': '01-agentic-rag/lessons/01-intro.md'}

In [38]:
usages

[CompletionUsage(completion_tokens=101, prompt_tokens=1063, total_tokens=2417, completion_tokens_details=None, prompt_tokens_details=None),
 CompletionUsage(completion_tokens=122, prompt_tokens=1400, total_tokens=2707, completion_tokens_details=None, prompt_tokens_details=None),
 CompletionUsage(completion_tokens=125, prompt_tokens=1886, total_tokens=3000, completion_tokens_details=None, prompt_tokens_details=None)]

In [44]:
import numpy as np

input_tokens = [u.prompt_tokens for u in usages]

print("Input tokens avg: ",np.mean(input_tokens))

Input tokens avg:  1449.6666666666667


**Answer:** 1400

# Searching the chunks

In [7]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [11]:
from embedder import Embedder

embed = Embedder()

np.float64(-0.02058203437252893)

In [42]:
from minsearch import VectorSearch, Index

chunks_content = [chunk["content"] for chunk in chunks]

# VectorSearch Index
X = embed.encode_batch(chunks_content)

vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks)

# TextSearch Index
index = Index(
    text_fields=['content'],
    keyword_fields=['filename']
)

index.fit(chunks)

In [52]:
def text_search(query, num_results=5):
    search_results = index.search(
        query,
        num_results=num_results
    )

    return search_results

def vector_search(query, num_results=5):
    query_vector = embed.encode(query)

    search_results = vindex.search(query_vector, num_results=num_results)

    return search_results

def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

# Q2. First result with text search

In [53]:
import pandas as pd

df_ground_truth = pd.read_csv("ground-truth.csv")
ground_truth = df_agent.to_dict(orient="records")

In [54]:
q = ground_truth[0]["question"]

q_search = text_search(q)

print(q_search[0]['filename'])


01-agentic-rag/lessons/03-rag.md


# Q3. First result with vector search

In [55]:
q_vsearch = vector_search(q)

print(q_vsearch[0]['filename'])

01-agentic-rag/lessons/01-intro.md


# Evaluation metrics

In [56]:
from tqdm.auto import tqdm

def compute_relevance(q, search_function):
    doc_id = q["filename"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == doc_id))

    return relevance

def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

# Q4. Evaluating text search

In [57]:
results = evaluate(ground_truth, text_search)

print(results)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.7583333333333333, 'mrr': 0.5942592592592594}


# Q5. Evaluating vector search

In [58]:
results = evaluate(ground_truth, vector_search)

print(results)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.725, 'mrr': 0.5486111111111112}


# Q6. Tuning hybrid search

In [62]:
for k in [1, 50, 100, 200]:
    print(f"Evaluating for k = {k}")

    def hybrid_search_k(query, k=k):
        text_results = text_search(query, num_results=10)
        vector_results = vector_search(query, num_results=10)
        return rrf([text_results, vector_results], k=k)
        
    results = evaluate(ground_truth, hybrid_search_k)
    print(f"hit_rate = {results['hit_rate']:.4f}, mrr = {results['mrr']:.4f}") 

Evaluating for k = 1


  0%|          | 0/360 [00:00<?, ?it/s]

hit_rate = 0.8389, mrr = 0.6482
Evaluating for k = 50


  0%|          | 0/360 [00:00<?, ?it/s]

hit_rate = 0.8361, mrr = 0.6379
Evaluating for k = 100


  0%|          | 0/360 [00:00<?, ?it/s]

hit_rate = 0.8361, mrr = 0.6379
Evaluating for k = 200


  0%|          | 0/360 [00:00<?, ?it/s]

hit_rate = 0.8361, mrr = 0.6379
